# Task 2: Sentiment Classification on Yelp Polarity

This notebook trains three models from scratch (baseline, BiLSTM, BiLSTM with attention). The config name is read from the `CONFIG` environment variable (`local`, `gpu` or `gpu_200k`).

## 1. Setup

In [ ]:
import json
import os
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yaml
from sklearn.metrics import accuracy_score, f1_score

sys.path.append("src")
from data import ROOT, clean_text, get_data, load_config, load_test_raw
from models import build_model, count_parameters
from utils import (Logger, get_device, load_checkpoint, make_scheduler, peak_memory_mb,
                   reset_peak_memory, save_checkpoint)

config_name = os.environ.get("CONFIG", "local")
cfg = load_config(config_name)

random.seed(cfg["seed"])
np.random.seed(cfg["seed"])
torch.manual_seed(cfg["seed"])

device, device_name = get_device()

out_dir = ROOT / cfg["paths"]["output_dir"]
out_dir.mkdir(parents=True, exist_ok=True)
print("config:", config_name)
print("device:", device_name)

## 2. Load Data

In [ ]:
train_loader, val_loader, test_loader, word_to_idx, info = get_data(cfg)
print("train reviews:", info["counts"]["train"])
print("val reviews:  ", info["counts"]["val"])
print("test reviews: ", info["counts"]["test"])

## 3. Exploring the Data

### Class balance

In [ ]:
balance = pd.DataFrame(info["class_counts"]).T
balance["positive share"] = (balance["positive"] / (balance["positive"] + balance["negative"])).round(3)
print(balance)

balance[["negative", "positive"]].plot(kind="bar", figsize=(6, 4), rot=0)
plt.title("Class Counts per Split")
plt.xlabel("split")
plt.ylabel("reviews")
plt.savefig(out_dir / "class_balance.png", dpi=150, bbox_inches="tight")
plt.show()

### Missing or broken rows

In [ ]:
dropped = pd.DataFrame(info["dropped"]).T[["total", "not_text", "empty", "bad_label", "kept"]]
print(dropped)
print("empty after cleaning:", info["empty_after_cleaning"])

### Review length

In [ ]:
arrays = np.load(ROOT / cfg["paths"]["processed_dir"] / "arrays.npz")
words = arrays["train_words"]
p50, p90, p95 = np.percentile(words, [50, 90, 95])
print(f"words per review: 50th = {p50:.0f}, 90th = {p90:.0f}, 95th = {p95:.0f}")

plt.figure(figsize=(7, 4))
plt.hist(words, bins=50, range=(0, np.percentile(words, 99)), color="steelblue")
for value, name in [(p50, "50th"), (p90, "90th"), (p95, "95th")]:
    plt.axvline(value, color="red", linestyle="--")
    plt.text(value, plt.ylim()[1] * 0.95, f" {name}: {value:.0f}", rotation=90, va="top")
plt.title("Review Length in Words (train)")
plt.xlabel("words")
plt.ylabel("reviews")
plt.savefig(out_dir / "length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Cleaning and Tokenizing

### Before and after cleaning

In [ ]:
raw = load_test_raw(cfg)
for i in range(3):
    text = raw["text"].iloc[i]
    print(f"review {i + 1} (label {raw['label'].iloc[i]})")
    print("before:", text[:400])
    print("after: ", clean_text(text)[:400])
    print()

### Vocab and max length

In [ ]:
print("vocab size:", info["vocab_size"])
print("max length:", info["max_len"])
for name, share in info["cut_share"].items():
    print(f"{name}: {share * 100:.1f}% of reviews are cut at max length")

## 5. Training the Models

### Settings and validation

In [ ]:
tcfg = cfg["train"]
model_names = ["baseline", "bilstm", "bilstm_attn"]
vocab_size = len(word_to_idx)
loss_fn = nn.BCEWithLogitsLoss()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum, count = 0.0, 0
    preds, labels = [], []
    for ids, lengths, y in loader:
        ids, lengths, y = ids.to(device), lengths.to(device), y.to(device)
        logits = model(ids, lengths)
        loss_sum += loss_fn(logits, y.float()).item() * len(y)
        count += len(y)
        preds.append((logits > 0).long().cpu())
        labels.append(y.cpu())
    preds = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    model.train()
    return loss_sum / count, accuracy_score(labels, preds), f1_score(labels, preds, average="macro")

### Training function

In [ ]:
def train_model(name):
    run_dir = out_dir / name
    ckpt_dir = ROOT / cfg["paths"]["checkpoint_dir"] / name
    if (run_dir / "summary.json").exists():
        print(f"{name}: already finished, skipping")
        return
    run_dir.mkdir(parents=True, exist_ok=True)
    logger = Logger(run_dir / "train_log.txt")

    torch.manual_seed(cfg["seed"])
    mcfg = cfg["models"][name]
    model = build_model(name, cfg, vocab_size).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=mcfg["lr"], weight_decay=tcfg["weight_decay"])
    total_steps = tcfg["epochs"] * len(train_loader)
    scheduler = make_scheduler(optimizer, int(tcfg["warmup_ratio"] * total_steps), total_steps)

    logger.log(f"model: {name}")
    logger.log(f"device: {device_name}")
    logger.log(f"torch version: {torch.__version__}")
    logger.log("config:\n" + yaml.dump({"model": mcfg, "train": tcfg}, sort_keys=False))
    logger.log(f"parameters: {count_parameters(model)}")

    last_path, best_path = ckpt_dir / "last.pt", ckpt_dir / "best.pt"
    start_epoch, step, best_f1, bad_epochs = 0, 0, -1.0, 0
    history = []
    if last_path.exists():
        ckpt = load_checkpoint(last_path, model, optimizer, scheduler, device)
        start_epoch, step = ckpt["epoch"], ckpt["step"]
        best_f1, bad_epochs = ckpt["best_f1"], ckpt["bad_epochs"]
        history = pd.read_csv(run_dir / "history.csv").iloc[:start_epoch].to_dict("records")
        logger.log(f"resumed from last.pt at epoch {start_epoch}, step {step}")

    reset_peak_memory(device)
    model.train()
    for epoch in range(start_epoch, tcfg["epochs"]):
        if bad_epochs >= tcfg["patience"]:
            break
        torch.manual_seed(cfg["seed"] + epoch)
        epoch_start = time.time()
        interval_start = time.time()
        interval_loss, interval_n, interval_examples = 0.0, 0, 0
        epoch_loss, epoch_n, nan_count = 0.0, 0, 0

        for ids, lengths, y in train_loader:
            ids, lengths, y = ids.to(device), lengths.to(device), y.float().to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(ids, lengths), y)
            if not torch.isfinite(loss):
                nan_count += 1
                continue
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), tcfg["grad_clip"]).item()
            optimizer.step()
            scheduler.step()
            step += 1

            interval_loss += loss.item()
            interval_n += 1
            interval_examples += len(y)
            epoch_loss += loss.item()
            epoch_n += 1

            if step % tcfg["log_every"] == 0:
                speed = interval_examples / (time.time() - interval_start)
                logger.log(f"step {step} | train loss {interval_loss / interval_n:.4f} | "
                           f"lr {scheduler.get_last_lr()[0]:.6f} | grad norm {grad_norm:.3f} | "
                           f"examples/sec {speed:.0f}")
                interval_start = time.time()
                interval_loss, interval_n, interval_examples = 0.0, 0, 0

        train_time = time.time() - epoch_start
        train_loss = epoch_loss / epoch_n
        val_loss, val_acc, val_f1 = evaluate(model, val_loader)
        epoch_time = time.time() - epoch_start
        mem = peak_memory_mb(device)
        logger.log(f"epoch {epoch + 1}/{tcfg['epochs']} | train loss {train_loss:.4f} | "
                   f"val loss {val_loss:.4f} | val acc {val_acc:.4f} | val macro-F1 {val_f1:.4f} | "
                   f"time {epoch_time:.1f}s | nan losses {nan_count}")

        history.append({"epoch": epoch + 1, "step": step, "train_loss": train_loss,
                        "val_loss": val_loss, "val_acc": val_acc, "val_macro_f1": val_f1,
                        "train_time": train_time, "epoch_time": epoch_time, "nan_losses": nan_count,
                        "peak_memory_mb": mem if mem is not None else float("nan")})
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

        if val_f1 > best_f1:
            best_f1, bad_epochs = val_f1, 0
            save_checkpoint(best_path, model, optimizer, scheduler, epoch + 1, step, best_f1, bad_epochs, cfg)
        else:
            bad_epochs += 1
        save_checkpoint(last_path, model, optimizer, scheduler, epoch + 1, step, best_f1, bad_epochs, cfg)
        if bad_epochs >= tcfg["patience"]:
            logger.log(f"early stopping: val macro-F1 did not improve for {tcfg['patience']} epochs")

    h = pd.DataFrame(history)
    peak = h["peak_memory_mb"].max()
    summary = {
        "model": name,
        "device": device_name,
        "parameters": count_parameters(model),
        "epochs_run": len(h),
        "stopped_early": bad_epochs >= tcfg["patience"],
        "best_epoch": int(h.loc[h["val_macro_f1"].idxmax(), "epoch"]),
        "best_val_macro_f1": float(best_f1),
        "total_training_time_s": float(h["epoch_time"].sum()),
        "examples_per_sec": float(len(train_loader.dataset) * len(h) / h["train_time"].sum()),
        "peak_memory_mb": None if pd.isna(peak) else float(peak),
        "nan_losses": int(h["nan_losses"].sum()),
    }
    with open(run_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    logger.log(f"finished {name}, best val macro-F1 {best_f1:.4f}")

### Train all three models

In [ ]:
for name in model_names:
    train_model(name)

summaries = []
for name in model_names:
    with open(out_dir / name / "summary.json") as f:
        summaries.append(json.load(f))
pd.DataFrame(summaries).set_index("model")

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name in model_names:
    h = pd.read_csv(out_dir / name / "history.csv")
    line, = axes[0].plot(h["epoch"], h["train_loss"], marker="o", label=f"{name} train")
    axes[0].plot(h["epoch"], h["val_loss"], marker="s", linestyle="--", color=line.get_color(), label=f"{name} val")
    axes[1].plot(h["epoch"], h["val_macro_f1"], marker="o", label=name)
axes[0].set_title("Train vs Validation Loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[1].set_title("Validation Macro-F1")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("macro-F1")
for ax in axes:
    ax.legend()
    ax.grid(True)
plt.savefig(out_dir / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()